# 05 · Ringkasan: Tabel Komparasi & Figur Gabungan
**Deep Learning (D6421) – Sesi 07: Generative Models | Tugas Take-Home**

Tidak ada training di sini (cukup CPU, Internet tidak wajib). Notebook membaca hasil `01`–`04`, lalu membuat **Tabel Bagian IV**, **Figur 2, 4, 5**, Figur Pendukung S2, dan zip seluruh output.

**Sebelum run:** tambahkan output keempat notebook sebagai Input: *Add Input → Your Work / Notebook Output* → pilih `01_autoencoders`, `02_gans`, `03_ddpm`, `04_mlp_limitations` (versi yang sudah di-commit). Pakai mode `QUICK_TEST` yang sama dengan run keempat notebook itu.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import copy
import csv
import glob
import json
import math
import random
import shutil
import statistics
import time
from dataclasses import asdict, dataclass
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, Markdown, display
from matplotlib.gridspec import GridSpec
from torch.utils.data import DataLoader, TensorDataset

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
@dataclass
class Config:
    # ---- Umum ----
    seed: int = 42
    data_root: str = "./data"
    out_dir: str = "./outputs"
    batch_size: int = 128
    num_workers: int = 0              # data di-preload ke tensor -> 0 sudah cepat
    fake_data: bool = False           # True = smoke test dengan data sintetis
    deterministic: bool = False
    train_frac: float = 0.8
    val_frac: float = 0.1

    # ---- Keluarga Autoencoder (backbone sama: 784-256-64-256-784) ----
    img_dim: int = 784
    hidden_dim: int = 256
    latent_dim: int = 64
    ae_epochs: int = 30
    ae_lr: float = 1e-3
    sparse_lambda: float = 1e-4       # lambda * ||h||_1
    sparse_sweep: Tuple[float, ...] = (0.0, 1e-5, 1e-4, 1e-3)
    sweep_epochs: int = 20
    vae_recon_loss: str = "bce"       # "bce" | "mse" (dijumlah per citra)
    vae_beta: float = 1.0

    # ---- GAN ----
    z_dim: int = 100
    gan_epochs: int = 60
    dcgan_epochs: int = 30
    gan_lr: float = 2e-4
    gan_beta1: float = 0.5
    gan_loss: str = "nonsat"          # "nonsat" | "minimax"
    dcgan_ngf: int = 64
    dcgan_ndf: int = 64
    gan_log_every: int = 10

    # ---- DDPM ----
    ddpm_epochs: int = 40
    ddpm_lr: float = 2e-4
    ddpm_T: int = 1000
    ddpm_beta_start: float = 1e-4
    ddpm_beta_end: float = 0.02
    ddpm_base_ch: int = 64
    ddpm_ch_mults: Tuple[int, ...] = (1, 2, 2)
    ddpm_num_res_blocks: int = 2
    ddpm_attn_res: Tuple[int, ...] = (7,)
    ddpm_dropout: float = 0.1
    ddpm_ema_decay: float = 0.999
    ddpm_amp: bool = True
    ddpm_sample_batch: int = 500
    ddpm_val_batches: int = 20

    # ---- MLP Classifier (Bagian 2): 784 -> 256 -> 64 -> 10 ----
    clf_epochs: int = 25
    clf_lr: float = 1e-3
    clf_dropout: float = 0.2
    clf_min_acc: float = 0.88
    latent_dec_epochs: int = 30       # = ae_epochs agar adil vs Vanilla AE

    # ---- Evaluasi ----
    fid_feature: int = 2048
    fid_n_samples: int = 0            # 0 = samakan dengan ukuran test set
    fid_batch: int = 100              # Inception resize ke 299x299 -> batch besar = OOM di T4
    latency_batch: int = 64
    latency_runs: int = 20
    latency_runs_ddpm: int = 3
    interp_classes: Tuple[int, int] = (1, 8)      # Trouser vs Bag
    interp_alphas: Tuple[float, ...] = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
    gan_interp_candidates: int = 2000

    def apply_fake_data_overrides(self):
        """Mode smoke test: semuanya kecil agar pipeline bisa dicek cepat di CPU."""
        self.ae_epochs = self.sweep_epochs = self.clf_epochs = self.latent_dec_epochs = 1
        self.gan_epochs = self.dcgan_epochs = self.ddpm_epochs = 1
        self.ddpm_T = 20
        self.ddpm_base_ch = 32
        self.ddpm_num_res_blocks = 1
        self.ddpm_sample_batch = 64
        self.ddpm_val_batches = 2
        self.fid_n_samples = 128
        self.latency_runs = 2
        self.latency_runs_ddpm = 1
        self.sparse_sweep = (0.0, 1e-4)
        self.clf_min_acc = 0.0
        self.gan_interp_candidates = 200


CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


GENERATIVE = ["vanilla_ae", "sparse_ae", "vae", "vanilla_gan", "dcgan", "ddpm"]


PRETTY = {"vanilla_ae": "Vanilla AE", "sparse_ae": "Sparse AE", "vae": "VAE",
          "vanilla_gan": "Vanilla GAN", "dcgan": "DCGAN", "ddpm": "DDPM",
          "mlp_classifier": "MLP Classifier", "latent_decoder": "Frozen-Latent Decoder"}

In [ ]:
def set_seed(seed=42, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path


def _json_default(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, tuple):
        return list(o)
    raise TypeError(f"Tidak bisa serialisasi {type(o)}")


def save_json(obj, path):
    ensure_dir(os.path.dirname(path) or ".")
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)


def to_01(x):
    """[-1, 1] -> [0, 1]"""
    return ((x + 1) / 2).clamp(0, 1)


def to_pm1(x):
    """[0, 1] -> [-1, 1]"""
    return x * 2 - 1


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def measure_latency(fn, device, n_warmup=3, n_runs=20):
    """Median & std latensi (ms) satu pemanggilan fn(). synchronize agar waktu GPU terhitung."""
    for _ in range(n_warmup):
        fn()
    _sync(device)
    times = []
    for _ in range(n_runs):
        _sync(device)
        t0 = time.perf_counter()
        fn()
        _sync(device)
        times.append((time.perf_counter() - t0) * 1000.0)
    return {"median_ms": statistics.median(times),
            "std_ms": statistics.pstdev(times) if len(times) > 1 else 0.0,
            "n_runs": n_runs}


def _free_cuda():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

In [ ]:
QUICK_TEST = True     # samakan dengan run notebook 01-04
FAKE_DATA = False

MODE = "trial" if (QUICK_TEST or FAKE_DATA) else "outputs"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./kaggle_working"
INPUT_ROOT = "/kaggle/input" if os.path.isdir("/kaggle/input") else "./kaggle_input"

cfg = Config(out_dir=os.path.join(BASE, MODE))
if FAKE_DATA:
    cfg.fake_data = True
    cfg.apply_fake_data_overrides()
for d in ["figures", "results"]:
    ensure_dir(os.path.join(cfg.out_dir, d))


def find_file(rel):
    """Cari file di output notebook ini dulu, lalu di semua Input (output notebook 01-04)."""
    own = os.path.join(cfg.out_dir, rel)
    if os.path.exists(own):
        return own
    hits = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", MODE, rel), recursive=True))
    return hits[0] if hits else None


def load_json_rel(rel):
    p = find_file(rel)
    return json.load(open(p)) if p else None


def load_pt_rel(rel):
    p = find_file(rel)
    return torch.load(p, map_location="cpu", weights_only=False) if p else None


def show_fig(name, width=1100):
    display(Image(filename=os.path.join(cfg.out_dir, "figures", name), width=width))


print("Output dir:", os.path.abspath(cfg.out_dir))
print("Folder Input yang terdeteksi:")
for p in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", MODE), recursive=True)):
    print("  ", p)

In [ ]:
def _save_grid(imgs, path, nrow=8):
    from torchvision.utils import save_image
    save_image(imgs.cpu(), path, nrow=nrow, padding=2, pad_value=1.0)


def _grid_np(imgs, nrow, pad_value=1.0):
    from torchvision.utils import make_grid
    return make_grid(imgs.cpu(), nrow=nrow, padding=2, pad_value=pad_value)[0].numpy()


def _savefig(fig, cfg, name):
    path = os.path.join(cfg.out_dir, "figures", name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    log(f"Figur disimpan: {path}")
    return path


def _image_rows(rows, row_labels, col_labels, title, figsize_per=(1.1, 1.15)):
    n_rows, n_cols = len(rows), rows[0].shape[0]
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * figsize_per[0] + 1.5, n_rows * figsize_per[1]),
                             squeeze=False)
    for i, (r, lab) in enumerate(zip(rows, row_labels)):
        for j in range(n_cols):
            ax = axes[i, j]
            ax.imshow(r[j, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=9)
            if i == 0 and col_labels is not None:
                ax.set_title(col_labels[j], fontsize=8)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    return fig


def _fmt(d):
    return {"MSE": round(d["mse"], 5), "PSNR (dB)": round(d["psnr"], 2), "SSIM": round(d["ssim"], 4)}


def _write_table(rows, path_no_ext):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path_no_ext + ".csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    flat = lambda v: f"{v:.4g}" if isinstance(v, float) else str(v)
    md = ["| " + " | ".join(keys) + " |", "|" + "---|" * len(keys)]
    md += ["| " + " | ".join(flat(r[k]) for k in keys) + " |" for r in rows]
    with open(path_no_ext + ".md", "w") as f:
        f.write("\n".join(md) + "\n")
    print("\n".join(md), flush=True)


def _smooth(v, k=25):
    v = np.asarray(v, dtype=float)
    if len(v) < k:
        return v
    return np.convolve(v, np.ones(k) / k, mode="valid")


def figure2_sampling(samples, cfg):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
    for ax, (lab, name) in zip(axes, [("A. VAE  z~N(0,I)", "vae"), ("B. Vanilla GAN  z~N(0,I)", "vanilla_gan"),
                                      ("C. DCGAN  z~N(0,I)", "dcgan"), ("D. DDPM  x_T~N(0,I)", "ddpm")]):
        ax.axis("off")
        if name in samples:
            ax.imshow(_grid_np(samples[name][:16], 4), cmap="gray", vmin=0, vmax=1)
        ax.set_title(lab, fontsize=11)
    fig.suptitle("Figur 2. Random Generative Sampling (4x4 per model)", fontsize=13)
    fig.tight_layout()
    return _savefig(fig, cfg, "fig2_random_sampling.png")


def figure4_interpolation(rows, cfg):
    order = [n for n in ["vanilla_ae", "sparse_ae", "vae", "vanilla_gan", "dcgan"] if n in rows]
    cA, cB = cfg.interp_classes
    fig = _image_rows([rows[n] for n in order], [PRETTY[n] for n in order],
                      [f"α={a:.1f}" for a in cfg.interp_alphas],
                      f"Figur 4. Linear Latent Interpolation: {CLASS_NAMES[cA]} → {CLASS_NAMES[cB]}",
                      figsize_per=(1.3, 1.35))
    return _savefig(fig, cfg, "fig4_latent_interpolation.png")


def figure5_training(hists, trajectory, cfg):
    fig = plt.figure(figsize=(17, 12))
    gs = GridSpec(3, 3, figure=fig, height_ratios=[1, 1, 1.05])

    def curve(ax, h, keys, title, ylabel):
        for k, lab in keys:
            if k in h:
                mk = "o" if len(h[k]) <= 15 else None
                ax.plot(range(1, len(h[k]) + 1), h[k], label=lab, marker=mk, ms=3)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 0])
    curve(ax, hists.get("vanilla_ae", {}), [("train_loss", "train"), ("val_loss", "val")], "Vanilla AE", "MSE")
    ax = fig.add_subplot(gs[0, 1])
    curve(ax, hists.get("sparse_ae", {}), [("train_loss", "train total"), ("val_loss", "val total"),
                                           ("val_rec", "val MSE")], "Sparse AE", "MSE + λ||h||₁")
    ax = fig.add_subplot(gs[0, 2])
    curve(ax, hists.get("vae", {}), [("train_loss", "train -ELBO"), ("val_loss", "val -ELBO"),
                                     ("val_reg", "val KL")], "VAE", "nats / citra")

    for col, name in [(0, "vanilla_gan"), (1, "dcgan")]:
        ax = fig.add_subplot(gs[1, col])
        h = hists.get(name, {})
        if h:
            le = h.get("log_every", 1)
            for k, lab, c in [("iter_g_loss", "Generator", "C1"), ("iter_d_loss", "Discriminator", "C3")]:
                raw = np.asarray(h[k])
                ax.plot(np.arange(len(raw)) * le, raw, color=c, alpha=0.2, lw=0.6)
                sm = _smooth(raw)
                ax.plot(np.arange(len(sm)) * le, sm, color=c, label=f"{lab} (smoothed)")
            ax.set_xlabel("Iterasi")
            ax.set_ylabel("BCE loss")
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)
        ax.set_title(f"{PRETTY[name]} (G vs D)")
    ax = fig.add_subplot(gs[1, 2])
    curve(ax, hists.get("ddpm", {}), [("train_loss", "train"), ("val_loss", "val (EMA)")],
          "DDPM (noise prediction)", "MSE(ε, ε_θ)")

    ax = fig.add_subplot(gs[2, :])
    ax.axis("off")
    if trajectory:
        imgs = torch.cat([to_01(x) for _, x in trajectory], 0)          # (steps*n,1,28,28)
        n = trajectory[0][1].shape[0]
        steps = len(trajectory)
        imgs = imgs.view(steps, n, 1, 28, 28).transpose(0, 1).reshape(-1, 1, 28, 28)
        ax.imshow(_grid_np(imgs, steps), cmap="gray", vmin=0, vmax=1)
        ax.set_title("DDPM reverse diffusion trajectory: " +
                     "  ".join(f"t={t}" for t, _ in trajectory) + "   (kiri: x_T → kanan: x_0)", fontsize=10)
    fig.suptitle("Figur 5. Training Loss Curves & DDPM Denoising Trajectory", fontsize=14)
    fig.tight_layout()
    return _savefig(fig, cfg, "fig5_training_and_trajectory.png")


def figure_extra_prior_compare(mlp_tensors, vae_samples, cfg):
    """Pendukung Q4/Q5: sampling Gaussian di laten MLP vs Vanilla AE vs prior VAE."""
    panels = [("z_MLP ~ N(μ,σ²) → Frozen Decoder", mlp_tensors["prior_samples_all"]),
              ("z_AE ~ N(μ,σ²) → Vanilla AE decoder", mlp_tensors["ae_gaussian_samples"]),
              ("z ~ N(0,I) → VAE decoder", vae_samples)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.3))
    for ax, (t, s) in zip(axes, panels):
        ax.imshow(_grid_np(s[:36], 6), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        ax.set_title(t, fontsize=10)
    fig.suptitle("Figur Pendukung. Prior Sampling: ruang laten tanpa vs dengan regularisasi P(z)", fontsize=12)
    fig.tight_layout()
    return _savefig(fig, cfg, "figS2_prior_sampling_compare.png")

## 2. Cek kelengkapan hasil

In [ ]:
needed = {"01_autoencoders": ["results/vanilla_ae.json", "results/sparse_ae.json", "results/vae.json",
                              "samples/interp_rows_ae.pt"],
          "02_gans": ["results/vanilla_gan.json", "results/dcgan.json", "samples/interp_rows_gan.pt"],
          "03_ddpm": ["results/ddpm.json", "samples/ddpm_trajectory.pt"],
          "04_mlp_limitations": ["results/mlp_experiment.json", "samples/mlp_tensors.pt"]}
for nb, fs in needed.items():
    miss = [f for f in fs if find_file(f) is None]
    print(f"{'OK     ' if not miss else 'BELUM  '} {nb}" + (f"  -> kurang: {miss} (sudah ditambahkan sebagai Input?)" if miss else ""))

## 3. Tabel Bagian IV — Komparasi kuantitatif

In [ ]:
results = {n: r for n in GENERATIVE if (r := load_json_rel(f"results/{n}.json")) is not None}

rows = []
for name in GENERATIVE:
    if name not in results:
        continue
    r = results[name]
    lat = r.get("latency_reconstruct") if name in ("vanilla_ae", "sparse_ae") else r.get("latency_generate")
    rows.append({
        "Arsitektur": r["model"],
        "Dim Laten": r["latent_dim"],
        "Params": r["params"],
        "Train Time/Epoch (s)": round(r["train_time_per_epoch_s"], 2),
        "Latency (ms/batch64)": round(lat["median_ms"], 2) if lat else "N/A",
        "Latency mode": "reconstruct" if name in ("vanilla_ae", "sparse_ae") else "generate",
        "Test MSE": round(r["mse"], 5) if "mse" in r else "N/A",
        "Test PSNR (dB)": round(r["psnr"], 2) if "psnr" in r else "N/A",
        "Test SSIM": round(r["ssim"], 4) if "ssim" in r else "N/A",
        "FID": round(r["fid"], 2) if "fid" in r else "N/A",
        "Class entropy (coverage)": round(r["coverage"]["class_entropy_norm"], 3) if "coverage" in r else "N/A",
        "Loss CV tail (stabilitas)": round(r["loss_cv_tail"], 4),
    })
_write_table(rows, os.path.join(cfg.out_dir, "results", "metrics_table"))
fid_ns = {r["fid_n_samples"] for r in results.values() if "fid_n_samples" in r}
if len(fid_ns) > 1:
    print("PERINGATAN: jumlah sampel FID berbeda antar model:", fid_ns, "-> FID tidak sebanding!")
p = find_file("results/mlp_reconstruction_table.md")
if p:
    print("\nRekonstruksi MLP (Tahap 3-4):")
    display(Markdown(open(p).read()))

## 4. Figur 2 — Random generative sampling

In [ ]:
samples = {n: s for n in ["vae", "vanilla_gan", "dcgan", "ddpm"] if (s := load_pt_rel(f"samples/{n}.pt")) is not None}
figure2_sampling(samples, cfg)
show_fig("fig2_random_sampling.png")

## 5. Figur 4 — Linear latent interpolation

In [ ]:
interp_rows = {}
for f in ["samples/interp_rows_ae.pt", "samples/interp_rows_gan.pt"]:
    d = load_pt_rel(f)
    if d is not None:
        interp_rows.update(d)
figure4_interpolation(interp_rows, cfg)
show_fig("fig4_latent_interpolation.png", 800)
for f in ["results/interp_ae.json", "results/interp_gan.json"]:
    for n, d in (load_json_rel(f) or {}).items():
        print(f"{PRETTY[n]:12s} max/mean jump = {d['max_over_mean_jump']:.2f}",
              "| nn_dist_ratio =", [round(v, 2) for v in d.get("nn_dist_ratio", [])])

## 6. Figur 5 — Training loss curves & trajektori DDPM

In [ ]:
hists = {n: h for n in GENERATIVE if (h := load_json_rel(f"logs/{n}_history.json")) is not None}
traj = load_pt_rel("samples/ddpm_trajectory.pt") or []
figure5_training(hists, traj, cfg)
show_fig("fig5_training_and_trajectory.png", 1200)

## 7. Figur Pendukung S2 — Prior sampling tanpa vs dengan regularisasi P(z)

In [ ]:
mlp_tensors = load_pt_rel("samples/mlp_tensors.pt")
if mlp_tensors is not None and "vae" in samples:
    figure_extra_prior_compare(mlp_tensors, samples["vae"], cfg)
    show_fig("figS2_prior_sampling_compare.png")
else:
    print("Butuh output notebook 01 (sampel VAE) dan 04 (mlp_tensors) sebagai Input.")

## 8. Kumpulkan semua hasil & zip
Figur, tabel, dan log dari notebook 01–04 disalin ke output notebook ini (checkpoint tidak ikut supaya ukuran kecil), lalu di-zip. File zip muncul di panel *Output*.

In [ ]:
marker = os.sep + MODE + os.sep
for p in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", MODE, "**", "*"), recursive=True)):
    if not os.path.isfile(p):
        continue
    rel = p.rsplit(marker, 1)[1]                      # mis. figures/fig1_reconstruction.png
    if rel.split(os.sep)[0] not in ("figures", "results", "logs"):
        continue
    dst = os.path.join(cfg.out_dir, rel)
    if not os.path.exists(dst):
        ensure_dir(os.path.dirname(dst))
        shutil.copy2(p, dst)
for f in sorted(glob.glob(os.path.join(cfg.out_dir, "figures", "*.png"))):
    print(os.path.basename(f))
zip_path = shutil.make_archive(os.path.join(BASE, f"{MODE}_bundle"), "zip", cfg.out_dir)
print("\nTersimpan:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")